In [ ]:
#load environment variables
from dotenv import load_dotenv
load_dotenv("../.env")

In [4]:
#to read environment variables
import os
from pathlib import Path

#reads pdf files and extracts text page by page
import fitz

#break long text into overlapping chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

api_key = os.getenv("GOOGLE_API_KEY")
print(f"API key loaded: {api_key[:8]}...")

API key loaded: AQ.Ab8RN...


In [5]:
PAPERS_DIR = Path("../papers")
CHROMA_DIR = Path("../chroma_db")

# This function takes a PDF file path and returns a list of LangChain Document objects
# One Document per page — each Document holds the page text and metadata
def load_pdf(pdf_path: Path) -> list[Document]:

    pdf = fitz.open(str(pdf_path))

    #This list will hold one Document per page
    pages = []

    #loop through every page in the PDF
    for page_num in range(len(pdf)):

        #extract all text from this page
        text = pdf[page_num].get_text()

        #skip pages that are empty or nearly empty
        if len(text.strip()) < 50:
            continue

        #create a LangChain Document with the page text and useful metadata
        doc = Document(
            page_content=text,
            metadata={
                "source": pdf_path.stem,
                "page": page_num + 1
            }
        )
        pages.append(doc)

    return pages

#Load all 12 papers
all_pages = []

for pdf_file in sorted(PAPERS_DIR.glob("*.pdf")):                   #find every pdf in the papers
    pages = load_pdf(pdf_file)                                      #extract pages from this PDF
    all_pages.extend(pages)                                         #add them to the master list
    print(f"Loaded {len(pages)} pages from: {pdf_file.stem}")

print(f"\nTotal pages loaded: {len(all_pages)}")

Loaded 15 pages from: adam_optimizer
Loaded 15 pages from: attention_is_all_you_need
Loaded 11 pages from: batch_normalization
Loaded 16 pages from: bert
Loaded 13 pages from: deep_rl_atari
Loaded 18 pages from: dropout
Loaded 9 pages from: gan
Loaded 19 pages from: rag
Loaded 12 pages from: resnet
Loaded 30 pages from: scaling_laws
Loaded 22 pages from: vision_transformer
Loaded 7 pages from: word2vec

Total pages loaded: 187


In [7]:
# Create a text splitter
# chunk_size=1000 means each chunk is at most 1000 characters
# chunk_overlap=200 means the last 200 characters of one chunk are repeated at the start of the next
# Why overlap? So that a sentence split across two chunks isn't lost from either one
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200
)

#split every page document into smaller chunk Documents
all_chunks = splitter.split_documents(all_pages)

print(f"Total chunks created: {len(all_chunks)}")
print(f"\nExample chunk from '{all_chunks[0].metadata['source']}', page {all_chunks[0].metadata['page']}:")
print("-"*60)
print(all_chunks[0].page_content)

Total chunks created: 826

Example chunk from 'adam_optimizer', page 1:
------------------------------------------------------------
Published as a conference paper at ICLR 2015
ADAM: A METHOD FOR STOCHASTIC OPTIMIZATION
Diederik P. Kingma*
University of Amsterdam, OpenAI
dpkingma@openai.com
Jimmy Lei Ba∗
University of Toronto
jimmy@psi.utoronto.ca
ABSTRACT
We introduce Adam, an algorithm for ﬁrst-order gradient-based optimization of
stochastic objective functions, based on adaptive estimates of lower-order mo-
ments. The method is straightforward to implement, is computationally efﬁcient,
has little memory requirements, is invariant to diagonal rescaling of the gradients,
and is well suited for problems that are large in terms of data and/or parameters.
The method is also appropriate for non-stationary objectives and problems with
very noisy and/or sparse gradients. The hyper-parameters have intuitive interpre-
tations and typically require little tuning. Some connections to related a

In [12]:
import time
from langchain_google_genai import GoogleGenerativeAIEmbeddings
import chromadb

#set up the Gemini embedding model
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001"
)

#create a raw ChromaDB client that saves to disk
chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))

# Create (or get if already exists) a collection — think of this like a table in a database
collection = chroma_client.get_or_create_collection(name="ml_papers")
print(f"Total chunks to embed: {len(all_chunks)}")
print("Embedding in batches of 80 with a 65-second pause between batches...\n")

# We process 80 chunks at a time — safely under the 100/min free tier limit
BATCH_SIZE = 80
for i in range(0, len(all_chunks), BATCH_SIZE):

    #slice out this batch of chunks
    batch = all_chunks[i : i + BATCH_SIZE]

    #Extract the raw text from each LangChain Document
    texts = [doc.page_content for doc in batch]

    #extract the metadata from each document
    metadatas = [doc.metadata for doc in batch]

    #create a unique ID for each chunk
    ids = [f"chunk_{i + j}" for j in range(len(batch))]

    #send the texts to Gemini to get vectors back
    vectors = embeddings.embed_documents(texts)

    #store the vectors, text, metadata into ChromaDB
    collection.add(
        embeddings=vectors,
        documents=texts,
        metadatas=metadatas,
        ids=ids
    )

    print(f"Stored batch {i // BATCH_SIZE + 1} - chunks {i} to {i + len(batch)}")

    #if there are more batches to go, wait 65 secs before the next one
    if i + BATCH_SIZE < len(all_chunks):
        print("Waiting 65 seconds to respect the free tier rate limit...")
        time.sleep(65)

print(f"\nDone! ChromaDB saved to: {CHROMA_DIR.resolve()}")
print(f"Total vectors stored: {collection.count()}")

Total chunks to embed: 826
Embedding in batches of 80 with a 65-second pause between batches...

Stored batch 1 - chunks 0 to 80
Waiting 65 seconds to respect the free tier rate limit...
Stored batch 2 - chunks 80 to 160
Waiting 65 seconds to respect the free tier rate limit...
Stored batch 3 - chunks 160 to 240
Waiting 65 seconds to respect the free tier rate limit...
Stored batch 4 - chunks 240 to 320
Waiting 65 seconds to respect the free tier rate limit...
Stored batch 5 - chunks 320 to 400
Waiting 65 seconds to respect the free tier rate limit...
Stored batch 6 - chunks 400 to 480
Waiting 65 seconds to respect the free tier rate limit...
Stored batch 7 - chunks 480 to 560
Waiting 65 seconds to respect the free tier rate limit...
Stored batch 8 - chunks 560 to 640
Waiting 65 seconds to respect the free tier rate limit...
Stored batch 9 - chunks 640 to 720
Waiting 65 seconds to respect the free tier rate limit...
Stored batch 10 - chunks 720 to 800
Waiting 65 seconds to respect the 